# AI photo audit: GVI and line of sight from seat photos

This notebook is Module 2 of the Seat-Level Visibility Toolkit. For each photo (one photo per seat) it does the following:

1. Crops the photo to a 60° × 60° view (the angles are set in the settings cell).
2. Runs an AI segmentation model and keeps the pixels labeled as plants.
3. Computes the **GVI** = plant pixels ÷ all pixels in the cropped view × 100.
4. Checks **line of sight from the photo** = whether a single plant region is larger than the minimum size you set.
5. Writes a table with one row per seat (CSV file) and overlay images that show the plant pixels in green, so you can check them by eye.

**How to use it:** run the cells one at a time from top to bottom (press ▶ or Shift+Enter). Read the text above each cell first. The only cell you need to edit is the settings cell.

**Before you start**
- Uploaded photos are sent to Google Colab's servers. Make sure you have permission to photograph and upload images of this library, and that there are no people in the photos.
- Every default value (60° angles, 1% minimum plant size, and so on) is a project assumption, not a value set by WELL or LEED.
- The AI output must be checked against the overlay images and against hand-labeled masks before you rely on it (helper cells are at the end of this notebook).
- This is a research tool, not an official WELL or LEED checker.

## Step 0: Prepare the photos

- Name each photo with its seat ID, for example `A01.jpg`, `A02.jpg`. The file name becomes the seat ID in the results table.
- Take every photo at seated eye height, facing the direction the seat faces, on a tripod, with the same lens and no zoom.
- Put all the photos in one zip file, for example `photos.zip`.
- Supported formats: .jpg, .png, .heic (iPhone), .webp, .tif. If Step 3 says no image files were found, check the list of file names it prints.
- If you shoot on a phone, do not edit or crop the photos beforehand. The original file usually keeps the focal length in its EXIF data, which is used to compute the crop.

## Step 1: Install the libraries

This cell installs `transformers` (loads the model) and `pillow-heif` (lets Python read iPhone .HEIC photos). It takes a few tens of seconds. Colab already includes PyTorch.

**Recommended:** switch on a GPU from the menu *Runtime → Change runtime type → T4 GPU*. It is much faster, but the notebook also works on CPU, only more slowly.

In [ ]:
!pip -q install transformers pillow-heif

## Step 2: Settings (edit only this cell)

| Setting | Meaning |
|---|---|
| `CROP_DEG_H`, `CROP_DEG_V` | Horizontal and vertical view angle to measure (default 60° × 60°, which is a project assumption) |
| `USE_EXIF` | Whether to read the 35 mm-equivalent focal length from the photo file |
| `FOCAL_35MM_DEFAULT` | 35 mm-equivalent focal length of the camera, used when EXIF has none. A phone's main lens is usually about 24–28; check the photo details or the phone's specifications |
| `PLANT_WORDS` | Names of the model's classes that count as "plant" |
| `MIN_PLANT_AREA_PCT` | How large, as a % of the cropped view, a single plant region must be to count as "visible". The 1% default is an assumption: set it deliberately and test how much the result changes when you vary it |
| `SEAT_THRESHOLD_PCT` | Share of seats that must pass (75%, as in the WELL M09 text) |
| `MAX_SIDE` | The cropped view is shrunk so its longest side is at most this many pixels before the AI runs (default 1024 keeps memory use down) |
| `MODEL_NAME` | Segmentation model. If the GPU runs out of memory or it is too slow, try the small one: `nvidia/segformer-b0-finetuned-ade-512-512` |

In [ ]:
CROP_DEG_H = 60
CROP_DEG_V = 60
USE_EXIF = True
FOCAL_35MM_DEFAULT = 26
PLANT_WORDS = ['plant', 'tree', 'flower', 'palm', 'grass']
MIN_PLANT_AREA_PCT = 1.0
SEAT_THRESHOLD_PCT = 75
MODEL_NAME = 'nvidia/segformer-b5-finetuned-ade-640-640'

MAX_SIDE = 1024   # shrink the cropped view to at most this many pixels on the long side (saves RAM; GVI barely changes)

PHOTO_DIR = '/content/photos'
OUT_DIR = '/content/output'

## Step 3: Upload the photos

Run the cell and choose a zip file (or several image files) from your computer. The files are unpacked into the folder `/content/photos`.

In [ ]:
import os, shutil, zipfile
from pathlib import Path
from google.colab import files

try:
    from pillow_heif import register_heif_opener
    register_heif_opener()   # lets Python read .heic / .heif photos
except Exception as e:
    print('Could not enable HEIC support (.heic photos cannot be read):', e)

os.makedirs(PHOTO_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    if name.lower().endswith('.zip'):
        zipfile.ZipFile(name).extractall(PHOTO_DIR)
    else:
        shutil.move(name, os.path.join(PHOTO_DIR, name))

IMAGE_EXT = ('.jpg', '.jpeg', '.png', '.heic', '.heif', '.webp', '.tif', '.tiff', '.bmp')
all_files = [p for p in Path(PHOTO_DIR).rglob('*') if p.is_file() and '__MACOSX' not in str(p) and not p.name.startswith('._')]
photo_paths = sorted(p for p in all_files if p.suffix.lower() in IMAGE_EXT)
print(f'Files uploaded: {len(all_files)}   of which images: {len(photo_paths)}')
for p in photo_paths[:5]:
    print(' ', p.name)
if not photo_paths:
    print('No image files found. Files found:', [p.name for p in all_files][:10])
    raise RuntimeError('No image files found: check that the upload finished (wait for "Saving ..." to complete) and that the files are jpg/png/heic')

## Step 4: Functions that crop a photo to a 60° × 60° view

A phone photo is wider than 60° (usually about 65–80° horizontally), so the photo is cropped around its center to the angle you want. The calculation is:

1. Get the 35 mm-equivalent focal length (from EXIF, or from the setting).
2. Convert it to a focal length in pixels: `f_px = f35 × image diagonal (px) ÷ 43.27`.
3. Crop width = `2 × f_px × tan(angle ÷ 2)`, for both the horizontal and the vertical angle.

If the photo is narrower than the angle you asked for, the function crops as much as the photo has and flags the row (`angle_ok = False`). That means the numbers do not describe a true 60° view. The angles actually obtained are written in `actual_h_deg` and `actual_v_deg`.

**Important:** a 4:3 photo from a phone's main camera (35 mm-equivalent about 24–28) is usually about 65–70° wide but only about 50–55° tall, so it cannot cover 60° × 60°. There are two ways round this: shoot with the ultra-wide lens (0.5×), or measure a rectangular view instead, for example `CROP_DEG_V = 45` (the same angles must then be used at both sites and in Module 1).

In [ ]:
import math
import numpy as np
from PIL import Image, ImageOps

def get_focal35(path):
    """Read the 35 mm-equivalent focal length from EXIF; use the default if it is missing."""
    if USE_EXIF:
        try:
            with Image.open(path) as im:
                v = im.getexif().get_ifd(0x8769).get(0xA405)
            if v:
                return float(v), 'EXIF'
        except Exception:
            pass
    return float(FOCAL_35MM_DEFAULT), 'default'

def load_upright(path):
    im = ImageOps.exif_transpose(Image.open(path))
    return im.convert('RGB')

def crop_box(w, h, focal35):
    """Return the crop box (left, top, right, bottom) around the center, whether the full angle fits, and the angles actually obtained."""
    f_px = focal35 * math.hypot(w, h) / 43.27
    cw = 2 * f_px * math.tan(math.radians(CROP_DEG_H / 2))
    ch = 2 * f_px * math.tan(math.radians(CROP_DEG_V / 2))
    angle_ok = True
    if cw > w or ch > h:
        angle_ok = False
        cw, ch = min(cw, w), min(ch, h)
    cx, cy = w / 2, h / 2
    box = (int(round(cx - cw / 2)), int(round(cy - ch / 2)),
           int(round(cx + cw / 2)), int(round(cy + ch / 2)))
    # angles actually obtained (degrees), reported in the table
    ang_h = 2 * math.degrees(math.atan(cw / (2 * f_px)))
    ang_v = 2 * math.degrees(math.atan(ch / (2 * f_px)))
    return box, angle_ok, (ang_h, ang_v)

## Step 5: Load the AI segmentation model

This cell downloads a SegFormer model trained on the ADE20K dataset (indoor and outdoor scenes, with classes such as "plant", "tree", "grass", "flower" and "palm"). The first run takes a while, and the model files are several hundred megabytes.

The model assigns every pixel to a class. We keep the classes whose names match `PLANT_WORDS` and merge them into "plant". Check the printed list of chosen classes and make sure it is what you intend.

**Notes:** check the model's licence and terms of use before you publish anything based on it. The author of this notebook could not run the model download step while writing it, so if this cell shows an error, copy the message and ask for help.

In [ ]:
import torch
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

processor = SegformerImageProcessor.from_pretrained(MODEL_NAME)
model = SegformerForSemanticSegmentation.from_pretrained(MODEL_NAME).to(device).eval()

plant_ids = [int(i) for i, name in model.config.id2label.items()
             if name.split(',')[0].strip().lower() in PLANT_WORDS]
print('Classes counted as plant:')
for i in plant_ids:
    print(' ', i, model.config.id2label[i])

def segment_plants(img):
    """Take a PIL image and return a True/False array the size of the image (True = plant)."""
    inputs = processor(images=img, return_tensors='pt').to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        # upsample the output only to a small working size (at most 512 px on the long side) to save memory
        s = min(1.0, 512 / max(img.size))
        size = (max(1, round(img.height * s)), max(1, round(img.width * s)))
        up = torch.nn.functional.interpolate(logits, size=size, mode='bilinear', align_corners=False)
        pred = up.argmax(dim=1)[0].cpu().numpy()
    del logits, up, inputs
    if device == 'cuda':
        torch.cuda.empty_cache()
    small = Image.fromarray((np.isin(pred, plant_ids) * 255).astype('uint8'))
    return np.array(small.resize(img.size, Image.NEAREST)) > 127

## Step 6: Scoring functions

- **GVI** = share of pixels in the cropped view that are plant (as a %).
- **largest_plant_pct** = the largest single plant region (a group of touching pixels), as a % of the cropped view.
- **los_pass** = `largest_plant_pct` ≥ `MIN_PLANT_AREA_PCT`. If true, the seat counts as "seeing a plant" (line of sight from the photo).

This photo-based line-of-sight rule is the project's own approximation, not a definition from either standard.

In [ ]:
from scipy import ndimage

def score(mask):
    gvi = float(mask.mean() * 100)
    labels, n = ndimage.label(mask)
    if n:
        sizes = ndimage.sum(mask, labels, range(1, n + 1))
        largest = float(sizes.max() / mask.size * 100)
    else:
        largest = 0.0
    return gvi, largest, bool(largest >= MIN_PLANT_AREA_PCT)

def overlay(crop, mask):
    arr = np.array(crop).astype(float)
    green = np.array([0, 200, 0], dtype=float)
    arr[mask] = 0.5 * arr[mask] + 0.5 * green
    return Image.fromarray(arr.astype('uint8'))

## Step 7: Run every photo

If Colab says *session crashed*, all variables are lost and you must run the cells again from Step 2 (Runtime → Restart session and run all also works, but Step 3 will ask you to upload the photos again). If memory still runs out, lower `MAX_SIDE` to 768 or change `MODEL_NAME` to the small model `nvidia/segformer-b0-finetuned-ade-512-512`.

For each photo the loop does: load → crop → segment → score → save an overlay image in `/content/output/overlays`, then it collects everything into the table `/content/output/seat_scores.csv`.

Table columns: `seat_id`, `gvi_pct`, `largest_plant_pct`, `los_pass`, `focal35`, `focal_source` (EXIF or default), `angle_ok` (whether the photo was wide enough for the requested angle), `actual_h_deg` and `actual_v_deg` (the horizontal and vertical angles actually obtained).

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

os.makedirs(os.path.join(OUT_DIR, 'overlays'), exist_ok=True)
rows, store = [], {}
for p in tqdm(photo_paths):
    img = load_upright(p)
    f35, src = get_focal35(p)
    box, angle_ok, ang = crop_box(img.width, img.height, f35)
    crop = img.crop(box)
    crop.thumbnail((MAX_SIDE, MAX_SIDE))   # shrink before processing to save memory
    mask = segment_plants(crop)
    gvi, largest, los = score(mask)
    overlay(crop, mask).save(os.path.join(OUT_DIR, 'overlays', p.stem + '_overlay.jpg'))
    store[p.stem] = (crop, mask, box)
    rows.append(dict(seat_id=p.stem, gvi_pct=round(gvi, 1), largest_plant_pct=round(largest, 1),
                     los_pass=los, focal35=f35, focal_source=src, angle_ok=angle_ok,
                     actual_h_deg=round(ang[0], 1), actual_v_deg=round(ang[1], 1)))

results = pd.DataFrame(rows)
results.to_csv(os.path.join(OUT_DIR, 'seat_scores.csv'), index=False)
results

## Step 8: Look at the overlays

This shows the cropped photo (left) next to the same photo with plant pixels painted green (right) for the first six seats. Look for where the AI goes wrong, for example artificial plants, shadows, reflections in glass, green furniture, or plants that are counted although you cannot see them.

To see other seats, change `SHOW` to a list of seat IDs, for example `['A07', 'B02']`.

In [ ]:
import matplotlib.pyplot as plt

SHOW = list(store)[:6]
fig, axes = plt.subplots(len(SHOW), 2, figsize=(8, 3.2 * len(SHOW)))
axes = np.atleast_2d(axes)
for row, sid in zip(axes, SHOW):
    crop, mask, _ = store[sid]
    g = results.loc[results.seat_id == sid, 'gvi_pct'].iloc[0]
    row[0].imshow(crop); row[0].set_title(f'{sid} cropped'); row[0].axis('off')
    row[1].imshow(overlay(crop, mask)); row[1].set_title(f'{sid} GVI = {g}%'); row[1].axis('off')
plt.tight_layout()
plt.show()

## Step 9: Share of seats that pass

This counts the seats that pass the photo-based line-of-sight check and compares the share with the 75% threshold. The counting rule follows the project document: the number of passing seats must be at least `ceil(0.75 × number of seats counted)`.

Remember that the rooms and seats counted must follow the wording of the credit you are using, and that this result is not an official certification check.

In [ ]:
n = len(results)
if n == 0:
    raise RuntimeError('No results yet: run Step 3 (upload photos) and Step 7 successfully first')
need = math.ceil(SEAT_THRESHOLD_PCT / 100 * n)
passed = int(results.los_pass.sum())
print(f'Seats counted: {n}')
print(f'Pass the photo-based line-of-sight check: {passed} ({passed / n * 100:.0f}%)')
print(f'At least {need} seats must pass to reach {SEAT_THRESHOLD_PCT}%')
print('Against the threshold:', 'reached' if passed >= need else 'not reached')
if not results.angle_ok.all():
    print('Warning: some photos are narrower than the requested angle; see the angle_ok column in the table')
print('Mean GVI (%):', round(results.gvi_pct.mean(), 1), '  min:', results.gvi_pct.min(), '  max:', results.gvi_pct.max())

## Step 10: Sensitivity to the minimum plant size

This tries several values of `MIN_PLANT_AREA_PCT` and shows how the share of passing seats changes. If a small change in the setting moves the result a lot, then the conclusion depends on the setting itself and you must report that clearly.

In [ ]:
sens = []
for th in [0.1, 0.5, 1, 2, 5, 10]:
    ok = (results.largest_plant_pct >= th).sum()
    sens.append(dict(min_plant_size_pct=th, seats_passing=int(ok), share_passing_pct=round(ok / n * 100)))
pd.DataFrame(sens)

## Step 11: Download the results

This packs the CSV table and all overlay images into one zip file and downloads it to your computer.

In [ ]:
shutil.make_archive('/content/results', 'zip', OUT_DIR)
files.download('/content/results.zip')

## (Optional) Check accuracy against hand-labeled photos

Use this for the validation step in the project plan:

1. Paint the plants by hand on the **original** photo (not the cropped one), using a drawing or labelling tool such as Label Studio or an image editor. Export a black-and-white image the same size as the original photo: **plant = white, everything else = black**.
2. Name each mask file after the seat ID, for example `A01.png`, and put them in `masks.zip`.
3. Run this cell and upload `masks.zip`. The cell crops each hand-painted mask with the same box used for the photo, then computes the IoU (closer to 1 means closer agreement) and the difference in GVI.

The plan uses about 50 photos (a planning figure). A second person should also label some of them independently so that agreement can be measured (Cohen's kappa).

In [ ]:
mask_dir = '/content/masks'
shutil.rmtree(mask_dir, ignore_errors=True)   # remove masks from earlier runs so old files are not mixed in
os.makedirs(mask_dir, exist_ok=True)
up = files.upload()
for name in up:
    if name.lower().endswith('.zip'):
        zipfile.ZipFile(name).extractall(mask_dir)
    else:
        shutil.move(name, os.path.join(mask_dir, name))

mask_files = {p.stem: p for p in Path(mask_dir).rglob('*') if p.suffix.lower() in ('.png', '.jpg', '.jpeg') and '__MACOSX' not in str(p) and not p.name.startswith('._')}
ev = []
for sid, (crop, ai_mask, box) in store.items():
    if sid not in mask_files:
        continue
    m = ImageOps.exif_transpose(Image.open(mask_files[sid])).convert('L')
    m = m.crop(box)
    hand = np.array(m.resize(crop.size)) > 127
    inter = np.logical_and(hand, ai_mask).sum()
    union = np.logical_or(hand, ai_mask).sum()
    iou = inter / union if union else 1.0
    ev.append(dict(seat_id=sid, IoU=round(float(iou), 3),
                   gvi_hand=round(float(hand.mean() * 100), 1),
                   gvi_ai=round(float(ai_mask.mean() * 100), 1)))
if not ev:
    raise RuntimeError('No hand-painted masks matched the seat IDs. Mask files found: ' + str(sorted(mask_files)) + '; seat IDs: ' + str(sorted(store)) + '. Upload masks.zip when the upload box appears, and name each mask after its seat ID (for example IMG_8292.png).')
ev = pd.DataFrame(ev)
ev['gvi_diff'] = (ev.gvi_ai - ev.gvi_hand).round(1)
print('Photos compared:', len(ev), '  mean IoU:', round(ev.IoU.mean(), 3))
print('Mean GVI difference (AI - hand):', round(ev.gvi_diff.mean(), 1), '  standard deviation:', round(ev.gvi_diff.std(), 1))
ev.to_csv(os.path.join(OUT_DIR, 'validation_iou.csv'), index=False)
ev

## (Optional) Compare with a simple method: green color threshold

The project plan keeps a green color threshold as a baseline, to see whether the AI really does better. This cell counts pixels whose color lies in a green range (hue 70°–170°, not too pale, not too dark) and compares the resulting GVI with the AI's.

The color range is an assumed default; adjust it with `HUE_MIN`, `HUE_MAX`, `SAT_MIN` and `VAL_MIN`.

In [ ]:
from matplotlib.colors import rgb_to_hsv

HUE_MIN, HUE_MAX, SAT_MIN, VAL_MIN = 70 / 360, 170 / 360, 0.25, 0.15

def color_mask(img):
    hsv = rgb_to_hsv(np.array(img).astype(float) / 255)
    return (hsv[..., 0] >= HUE_MIN) & (hsv[..., 0] <= HUE_MAX) & (hsv[..., 1] >= SAT_MIN) & (hsv[..., 2] >= VAL_MIN)

base = []
for sid, (crop, ai_mask, box) in store.items():
    cm = color_mask(crop)
    base.append(dict(seat_id=sid, gvi_ai=round(float(ai_mask.mean() * 100), 1), gvi_color=round(float(cm.mean() * 100), 1)))
base = pd.DataFrame(base)
base['diff'] = (base.gvi_color - base.gvi_ai).round(1)
print('Mean difference (color - AI):', round(base['diff'].mean(), 1))
print('Correlation between the two methods:', round(base.gvi_ai.corr(base.gvi_color), 2))
base